# Notebook 03 — Feature Selection, Model Zoo, and Rigorous Metrics

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

### Why a broad model zoo?

Each algorithm family has a different inductive bias. A systematic
comparison is the honest way to discover which bias fits the task best:

| Family | Inductive bias | Strengths | Limitations |
|---|---|---|---|
| **OLS** | linear, no regularization | fast, interpretable | suffers from multicollinearity |
| **Ridge** ($L^2$) | linear + shrinkage | stable with collinear features | residual bias |
| **Lasso** ($L^1$) | linear + sparsity | automatic feature selection | picks arbitrarily among correlated features |
| **KNN** | local, distance-based | captures non-linearity | curse of dimensionality |
| **Decision Tree** | axis-aligned partitioning | captures interactions; interpretable | high variance |
| **Random Forest** | bagged trees | reduces variance | less efficient than boosting on subtle signals |
| **Gradient Boosting** | boosted trees | typically the strongest | overfits if under-regularized |
| **Gaussian NB** | conditional independence | fast, robust baseline | high bias |
| **Logistic** | log-linear | calibrated probabilities | linear |

### Non-negotiable baselines for meteorological time series

> *"Every forecast skill figure must be read against persistence.
> The model is only useful if it beats persistence."*
> — Wilks, *Statistical Methods in the Atmospheric Sciences*, 4th ed.

1. **Persistence-now**: $\hat{T}_{t+24} = T_t$
2. **Persistence-24h**: $\hat{T}_{t+24} = T_{t-24}$ (same hour yesterday — often the winner)
3. **Hourly climatology**: $\hat{T}_{t+24} = \overline{T}\,(c, h)$ — historical mean for city $c$ at hour $h$.
4. **Trivial-majority**: classifier that always predicts the modal class.

### Metrics

#### Regression (all in °C)

- **RMSE** = $\sqrt{\frac{1}{n}\sum(y-\hat{y})^2}$ — penalizes large errors
- **MAE** = $\frac{1}{n}\sum|y-\hat{y}|$ — robust to outliers
- **MedAE** — median absolute error
- **MAPE** = $\frac{100}{n}\sum\!\left|\frac{y-\hat{y}}{y}\right|$ — scale-free
- **MBE** = $\frac{1}{n}\sum(\hat{y}-y)$ — directional bias
- **MaxErr** — worst case
- **R²** = $1 - \frac{SS_{res}}{SS_{tot}}$ — variance explained

#### Classification

- **Accuracy** — $\frac{TP+TN}{N}$ (misleading on imbalanced data)
- **Balanced Accuracy** = $\frac{1}{2}\!\left(\frac{TP}{TP+FN} + \frac{TN}{TN+FP}\right)$
- **Precision** = $\frac{TP}{TP+FP}$
- **Recall** = $\frac{TP}{TP+FN}$
- **F1** = harmonic mean of precision and recall
- **MCC** (Matthews) — gold standard for imbalanced classes

#### Skill score vs baseline

$$
SS = 1 - \left(\frac{\text{RMSE}_{model}}{\text{RMSE}_{baseline}}\right)^2
$$

$SS > 0$ ⇒ model is better; $SS = 0$ ⇒ ties; $SS < 0$ ⇒ worse.

In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde", "approx"] }
:dep smartcore = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [2]:
use polars::prelude::*;
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis};
use std::collections::{HashMap, BTreeMap};
use std::fs::File;

use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::linear_regression::{LinearRegression, LinearRegressionParameters};
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};
use smartcore::linear::lasso::{Lasso, LassoParameters};
use smartcore::linear::logistic_regression::{LogisticRegression, LogisticRegressionParameters};
use smartcore::neighbors::knn_regressor::{KNNRegressor, KNNRegressorParameters};
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use smartcore::tree::decision_tree_regressor::{DecisionTreeRegressor, DecisionTreeRegressorParameters};
use smartcore::tree::decision_tree_classifier::{DecisionTreeClassifier, DecisionTreeClassifierParameters};
use smartcore::ensemble::random_forest_regressor::{RandomForestRegressor, RandomForestRegressorParameters};
use smartcore::ensemble::random_forest_classifier::{RandomForestClassifier, RandomForestClassifierParameters};
use smartcore::naive_bayes::gaussian::{GaussianNB, GaussianNBParameters};

println!("Dependencies loaded. Model zoo ready.");

Dependencies loaded. Model zoo ready.


---
## 1. Load splits from Notebook 02

In [3]:
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let val_df   = LazyFrame::scan_parquet("../data/features/val.parquet",   Default::default())
    .unwrap().collect().unwrap();
let test_df  = LazyFrame::scan_parquet("../data/features/test.parquet",  Default::default())
    .unwrap().collect().unwrap();

println!("Train: {:>5} x {}", train_df.height(), train_df.width());
println!("Val:   {:>5} x {}",   val_df.height(),   val_df.width());
println!("Test:  {:>5} x {}",  test_df.height(),  test_df.width());

Train: 20160 x 114


Val:    5040 x 114


Test:   5712 x 114


---
## 2. Define the feature set

We exclude:
- Metadata (`city`, `country_code`, `climate_zone`, `timestamp`)
- Raw variables already covered by log/lag (`precipitation` raw stays as
  a feature because the model needs the current state)
- Future targets (any `*_next_*` column, `weather_condition`)
- Discrete variables that become cyclical (`year`, `month`, `day`,
  `hour`, `day_of_week`, `day_of_year` — used only in cyclic encoding)
- `weathercode` (raw input encoded in `weather_condition` target)

The result is the set that does **not** leak future information.

In [4]:
// List every column
let all_cols: Vec<String> = train_df.get_column_names().iter().map(|s| s.to_string()).collect();
println!("Total columns in train: {}", all_cols.len());

let exclude: std::collections::HashSet<String> = [
    "city", "country_code", "climate_zone", "timestamp",
    "year", "month", "day", "hour", "day_of_week", "day_of_year",
    "weathercode",
    "temp_next_24h", "temp_next_48h", "temp_next_72h",
    "precip_sum_next_24h", "weather_condition", "will_rain_next_24h",
    "log1p_precip_next_24h",
].iter().map(|s| s.to_string()).collect();

let feature_cols: Vec<String> = all_cols.iter()
    .filter(|c| !exclude.contains(*c))
    .cloned().collect();

println!("Selected features: {}", feature_cols.len());
println!("First 10: {:?}", &feature_cols[..feature_cols.len().min(10)]);

Total columns in train: 114


Selected features: 96


First 10: ["latitude", "longitude", "elevation_m", "temperature_2m", "dewpoint_2m", "precipitation", "rain", "snowfall", "windspeed_10m", "windgusts_10m"]


---
## 3. DataFrame ↔ ndarray ↔ DenseMatrix helpers

`linfa` speaks ndarray, `smartcore` speaks `DenseMatrix`. Centralize the
conversion here to avoid drift.

In [5]:
fn df_to_array2(df: &DataFrame, cols: &[String]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for col_name in cols {
        let s = df.column(col_name.as_str()).unwrap();
        let f = s.cast(&DataType::Float64).unwrap();
        let ca = f.f64().unwrap().to_vec();
        for v in ca { data.push(v.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, name: &str) -> Array1<f64> {
    let v: Vec<f64> = df.column(name).unwrap()
        .cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec().into_iter()
        .map(|x| x.unwrap_or(0.0)).collect();
    Array1::from_vec(v)
}

fn ndarray_to_dense(a: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&a.outer_iter().map(|r| r.to_vec()).collect::<Vec<_>>())
}

println!("Helpers ready.");

Helpers ready.


---
## 4. Drop rows with null targets or critical features

Lags introduce nulls in the first observations of each city. To train on
valid data we remove rows where any target or the 24/48 h lag is null.

In [6]:
fn clean_rows(df: &DataFrame) -> DataFrame {
    df.clone().lazy().filter(
        col("temp_next_24h").is_not_null()
        .and(col("temp_next_48h").is_not_null())
        .and(col("temp_next_72h").is_not_null())
        .and(col("will_rain_next_24h").is_not_null())
        .and(col("temp_lag48h").is_not_null())
        .and(col("temp_lag24h").is_not_null())
    ).collect().unwrap()
}

let train_clean = clean_rows(&train_df);
let val_clean   = clean_rows(&val_df);
let test_clean  = clean_rows(&test_df);

println!("Valid rows:");
println!("  Train: {} (of {})", train_clean.height(), train_df.height());
println!("  Val:   {} (of {})", val_clean.height(),   val_df.height());
println!("  Test:  {} (of {})", test_clean.height(),  test_df.height());

Valid rows:


  Train: 19488 (of 20160)


  Val:   5040 (of 5040)


  Test:  4704 (of 5712)


In [7]:
let X_train = df_to_array2(&train_clean, &feature_cols);
let X_val   = df_to_array2(&val_clean,   &feature_cols);
let X_test  = df_to_array2(&test_clean,  &feature_cols);

// Targets - regression
let y_train_temp24 = df_to_array1(&train_clean, "temp_next_24h");
let y_val_temp24   = df_to_array1(&val_clean,   "temp_next_24h");
let y_test_temp24  = df_to_array1(&test_clean,  "temp_next_24h");

// Targets - classification
let y_train_rain: Vec<u32> = df_to_array1(&train_clean, "will_rain_next_24h").iter().map(|&v| v as u32).collect();
let y_val_rain:   Vec<u32> = df_to_array1(&val_clean,   "will_rain_next_24h").iter().map(|&v| v as u32).collect();
let y_test_rain:  Vec<u32> = df_to_array1(&test_clean,  "will_rain_next_24h").iter().map(|&v| v as u32).collect();

println!("Matrices:");
println!("  X_train: {:?}", X_train.shape());
println!("  X_val:   {:?}", X_val.shape());
println!("  X_test:  {:?}", X_test.shape());

Matrices:


  X_train: [19488, 96]


  X_val:   [5040, 96]


  X_test:  [4704, 96]


---
## 5. Variance threshold — drop near-constant features

Features with variance $< 10^{-8}$ carry no information and can break
matrix inversions. We do not expect to find many, but this filter is
defensive.

In [8]:
let n_features = X_train.ncols();
let mut keep_idx: Vec<usize> = Vec::new();
let mut drop_names: Vec<String> = Vec::new();

for j in 0..n_features {
    let col = X_train.column(j);
    let mean = col.mean().unwrap_or(0.0);
    let var: f64 = col.iter().map(|x| (x - mean).powi(2)).sum::<f64>() / (col.len() as f64 - 1.0);
    if var > 1e-8 {
        keep_idx.push(j);
    } else {
        drop_names.push(feature_cols[j].clone());
    }
}

println!("Features dropped for variance < 1e-8: {} {:?}", drop_names.len(), drop_names);
println!("Features kept: {}", keep_idx.len());

Features dropped for variance < 1e-8: 0 []


Features kept: 96


---
## 6. Multicollinearity analysis

We compute the $|R|$ matrix between features (absolute Pearson
correlation) and flag pairs with $|R| > 0.97$. For each pair we drop
the feature with the **lower correlation with the main target**
(`temp_next_24h`).

This is a pragmatic VIF approximation that avoids matrix inversion.

In [9]:
fn pearson_arr(x: &ArrayView1<f64>, y: &ArrayView1<f64>) -> f64 {
    let n = x.len() as f64;
    let mx = x.mean().unwrap_or(0.0);
    let my = y.mean().unwrap_or(0.0);
    let mut cov = 0.0; let mut vx = 0.0; let mut vy = 0.0;
    for (a, b) in x.iter().zip(y.iter()) {
        let dx = a - mx; let dy = b - my;
        cov += dx*dy; vx += dx*dx; vy += dy*dy;
    }
    if vx*vy < 1e-18 { 0.0 } else { cov / (vx*vy).sqrt() }
}

// Feature-to-target correlation - used to decide which one to drop.
let mut feat_target_corr: HashMap<usize, f64> = HashMap::new();
for &j in &keep_idx {
    let r = pearson_arr(&X_train.column(j), &y_train_temp24.view()).abs();
    feat_target_corr.insert(j, r);
}

// Identify collinear pairs
let mut to_drop: std::collections::HashSet<usize> = std::collections::HashSet::new();
let mut pairs_dropped: Vec<(String, String, f64)> = Vec::new();
for (idx_i, &i) in keep_idx.iter().enumerate() {
    if to_drop.contains(&i) { continue; }
    for &j in keep_idx.iter().skip(idx_i + 1) {
        if to_drop.contains(&j) { continue; }
        let r = pearson_arr(&X_train.column(i), &X_train.column(j)).abs();
        if r > 0.97 {
            // drop the one with lower target correlation
            let (drop, keep) = if feat_target_corr[&i] < feat_target_corr[&j] { (i, j) } else { (j, i) };
            to_drop.insert(drop);
            pairs_dropped.push((feature_cols[drop].clone(), feature_cols[keep].clone(), r));
        }
    }
}

let final_keep_idx: Vec<usize> = keep_idx.iter().filter(|&&i| !to_drop.contains(&i)).cloned().collect();
let final_features: Vec<String> = final_keep_idx.iter().map(|&i| feature_cols[i].clone()).collect();

println!("Collinear pairs (|R| > 0.97):");
for (d, k, r) in &pairs_dropped {
    println!("  drop  {:<25}  -> keep {:<25}  |r| = {:.3}", d, k, r);
}
println!();
println!("Features after decorrelation: {} (of {})", final_features.len(), feature_cols.len());

Collinear pairs (|R| > 0.97):


  drop  temp_lag1h                 -> keep temperature_2m             |r| = 0.994


  drop  td_lag1h                   -> keep dewpoint_2m                |r| = 0.997


  drop  td_lag6h                   -> keep dewpoint_2m                |r| = 0.974


  drop  interact_t_rh              -> keep dewpoint_2m                |r| = 0.982


...

  drop  log1p_windspeed            -> keep log_windspeed              |r| = 1.000


  drop  temp_roll24_min            -> keep temp_roll24_mean           |r| = 0.986


  drop  temp_roll24_max            -> keep temp_roll24_mean           |r| = 0.982


  drop  temp_diurnal_range_24h     -> keep temp_roll24_std            |r| = 0.975


Features after decorrelation: 80 (of 96)


In [10]:
let X_train_f = X_train.select(Axis(1), &final_keep_idx);
let X_val_f   = X_val.select(Axis(1),   &final_keep_idx);
let X_test_f  = X_test.select(Axis(1),  &final_keep_idx);

println!("Final matrices:");
println!("  X_train_f: {:?}", X_train_f.shape());
println!("  X_val_f:   {:?}", X_val_f.shape());
println!("  X_test_f:  {:?}", X_test_f.shape());

Final matrices:


  X_train_f: [19488, 80]


  X_val_f:   [5040, 80]


  X_test_f:  [4704, 80]


---
## 7. Standard scaling (z-score)

Distance-based models (KNN) and linear models (OLS, Ridge, Lasso,
Logistic) are sensitive to heterogeneous scales. We fit $\mu, \sigma$
on the **train** set and apply to val/test (no leakage).

In [11]:
let n_feat = X_train_f.ncols();
let mut means = vec![0.0; n_feat];
let mut stds  = vec![1.0; n_feat];
for j in 0..n_feat {
    let c = X_train_f.column(j);
    let mu = c.mean().unwrap_or(0.0);
    let var: f64 = c.iter().map(|x| (x - mu).powi(2)).sum::<f64>() / (c.len() as f64 - 1.0).max(1.0);
    means[j] = mu;
    stds[j]  = var.sqrt().max(1e-8);
}

fn standardize(x: &Array2<f64>, mu: &[f64], sd: &[f64]) -> Array2<f64> {
    let mut out = x.clone();
    for j in 0..x.ncols() {
        for i in 0..x.nrows() {
            out[[i, j]] = (x[[i, j]] - mu[j]) / sd[j];
        }
    }
    out
}

let X_train_z = standardize(&X_train_f, &means, &stds);
let X_val_z   = standardize(&X_val_f,   &means, &stds);
let X_test_z  = standardize(&X_test_f,  &means, &stds);

// DenseMatrix versions
let X_train_dm = ndarray_to_dense(&X_train_f);
let X_val_dm   = ndarray_to_dense(&X_val_f);
let X_test_dm  = ndarray_to_dense(&X_test_f);
let X_train_dm_z = ndarray_to_dense(&X_train_z);
let X_val_dm_z   = ndarray_to_dense(&X_val_z);
let X_test_dm_z  = ndarray_to_dense(&X_test_z);

println!("Standardization done ({} features scaled).", n_feat);

Standardization done (80 features scaled).


---
## 8. Top-k features by correlation with the regression target

In [12]:
let mut corr_temp: Vec<(String, f64)> = Vec::new();
for (idx, name) in final_features.iter().enumerate() {
    let r = pearson_arr(&X_train_f.column(idx), &y_train_temp24.view());
    corr_temp.push((name.clone(), r));
}
corr_temp.sort_by(|a, b| b.1.abs().partial_cmp(&a.1.abs()).unwrap());

println!("Top 20 features by |rho| with temp_next_24h:");
println!("{:<32} {:>10}", "feature", "rho");
for (name, r) in corr_temp.iter().take(20) {
    println!("{:<32} {:>10.4}", name, r);
}

Top 20 features by |rho| with temp_next_24h:


feature                                 rho


temperature_2m                       0.9629


temp_lag24h                          0.9348


temp_lag3h                           0.9318


...

shortwave_radiation                  0.4246


clear_sky_radiation                  0.3929


pres_roll24_max                     -0.3587


climate_Dfb                         -0.3587


()

In [13]:
let mut corr_rain: Vec<(String, f64)> = Vec::new();
let y_rain_f64: Array1<f64> = Array1::from_vec(y_train_rain.iter().map(|&v| v as f64).collect());
for (idx, name) in final_features.iter().enumerate() {
    let r = pearson_arr(&X_train_f.column(idx), &y_rain_f64.view());
    corr_rain.push((name.clone(), r));
}
corr_rain.sort_by(|a, b| b.1.abs().partial_cmp(&a.1.abs()).unwrap());

println!("Top 20 features by |rho| with will_rain_next_24h (point-biserial):");
println!("{:<32} {:>10}", "feature", "rho");
for (name, r) in corr_rain.iter().take(20) {
    println!("{:<32} {:>10.4}", name, r);
}

Top 20 features by |rho| with will_rain_next_24h (point-biserial):


feature                                 rho


climate_Csb                         -0.3729


cloudcover                           0.3300


climate_Cfb                          0.2076


...

vpd                                 -0.1161


precip_lag1h                         0.1160


month_sin                            0.1131


doy_sin                              0.1120


()

---
## 9. Metric functions (defined once, used throughout)

In [14]:
#[derive(Debug, Clone)]
struct RegMetrics {
    rmse: f64,
    mae:  f64,
    medae: f64,
    mape: f64,
    mbe:  f64,
    r2:   f64,
    max_err: f64,
}

fn reg_metrics(y_true: &[f64], y_pred: &[f64]) -> RegMetrics {
    let n = y_true.len() as f64;
    let mean_y = y_true.iter().sum::<f64>() / n;
    let mut se = 0.0; let mut ae = 0.0; let mut be = 0.0;
    let mut max_err = 0.0_f64;
    let mut ss_res = 0.0; let mut ss_tot = 0.0;
    let mut abs_err: Vec<f64> = Vec::with_capacity(y_true.len());
    let mut ape_sum = 0.0; let mut ape_n = 0.0;
    for (t, p) in y_true.iter().zip(y_pred.iter()) {
        let d = t - p;
        se += d * d;
        ae += d.abs();
        be += p - t;
        abs_err.push(d.abs());
        if d.abs() > max_err { max_err = d.abs(); }
        ss_res += d * d;
        ss_tot += (t - mean_y).powi(2);
        if t.abs() > 0.5 { ape_sum += (d / t).abs(); ape_n += 1.0; }
    }
    abs_err.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let medae = abs_err[abs_err.len()/2];
    RegMetrics {
        rmse: (se / n).sqrt(),
        mae:  ae / n,
        medae,
        mape: if ape_n > 0.0 { 100.0 * ape_sum / ape_n } else { 0.0 },
        mbe:  be / n,
        r2:   if ss_tot > 1e-18 { 1.0 - ss_res / ss_tot } else { 0.0 },
        max_err,
    }
}

#[derive(Debug, Clone)]
struct ClsMetrics {
    accuracy: f64,
    bal_acc:  f64,
    precision: f64,
    recall:    f64,
    f1:        f64,
    mcc:       f64,
    tn: usize, fp: usize, fn_: usize, tp: usize,
}

fn cls_metrics(y_true: &[u32], y_pred: &[u32]) -> ClsMetrics {
    let mut tp = 0usize; let mut tn = 0usize; let mut fp = 0usize; let mut fn_ = 0usize;
    for (t, p) in y_true.iter().zip(y_pred.iter()) {
        match (*t, *p) {
            (1, 1) => tp += 1,
            (0, 0) => tn += 1,
            (0, 1) => fp += 1,
            (1, 0) => fn_ += 1,
            _ => {}
        }
    }
    let total = (tp + tn + fp + fn_) as f64;
    let acc = (tp + tn) as f64 / total.max(1.0);
    let tpr = if tp + fn_ > 0 { tp as f64 / (tp + fn_) as f64 } else { 0.0 };
    let tnr = if tn + fp > 0 { tn as f64 / (tn + fp) as f64 } else { 0.0 };
    let prec = if tp + fp > 0 { tp as f64 / (tp + fp) as f64 } else { 0.0 };
    let rec = tpr;
    let f1 = if prec + rec > 0.0 { 2.0 * prec * rec / (prec + rec) } else { 0.0 };
    let denom = ((tp+fp) as f64 * (tp+fn_) as f64 * (tn+fp) as f64 * (tn+fn_) as f64).sqrt();
    let mcc = if denom > 0.0 { (tp as f64 * tn as f64 - fp as f64 * fn_ as f64) / denom } else { 0.0 };
    ClsMetrics { accuracy: acc, bal_acc: 0.5*(tpr+tnr), precision: prec, recall: rec, f1, mcc,
                 tn, fp, fn_, tp }
}

println!("Metric functions ready.");

Metric functions ready.


---
## 10. Baselines

### 10.1 Persistence-now: $\hat{T}_{t+24} = T_t$

In [15]:
// Persistence-now uses the current 'temperature_2m' column.
let p_now_train = df_to_array1(&train_clean, "temperature_2m");
let p_now_val   = df_to_array1(&val_clean,   "temperature_2m");
let p_now_test  = df_to_array1(&test_clean,  "temperature_2m");

let m_now_val  = reg_metrics(y_val_temp24.as_slice().unwrap(),  p_now_val.as_slice().unwrap());
let m_now_test = reg_metrics(y_test_temp24.as_slice().unwrap(), p_now_test.as_slice().unwrap());
println!("PERSISTENCE-now (val):  RMSE={:.3}  MAE={:.3}  R2={:.3}", m_now_val.rmse, m_now_val.mae, m_now_val.r2);
println!("PERSISTENCE-now (test): RMSE={:.3}  MAE={:.3}  R2={:.3}", m_now_test.rmse, m_now_test.mae, m_now_test.r2);

PERSISTENCE-now (val):  RMSE=2.415  MAE=1.746  R2=0.940


PERSISTENCE-now (test): RMSE=3.760  MAE=2.516  R2=0.835


### 10.2 Persistence-24h: $\hat{T}_{t+24} = T_{t-24}$ ("tomorrow will be like yesterday")

In [16]:
// Persistence-24h uses the 'temp_lag24h' column already built in Notebook 02.
let p_24_val   = df_to_array1(&val_clean,   "temp_lag24h");
let p_24_test  = df_to_array1(&test_clean,  "temp_lag24h");

let m_24_val  = reg_metrics(y_val_temp24.as_slice().unwrap(),  p_24_val.as_slice().unwrap());
let m_24_test = reg_metrics(y_test_temp24.as_slice().unwrap(), p_24_test.as_slice().unwrap());
println!("PERSISTENCE-24h (val):  RMSE={:.3}  MAE={:.3}  R2={:.3}", m_24_val.rmse, m_24_val.mae, m_24_val.r2);
println!("PERSISTENCE-24h (test): RMSE={:.3}  MAE={:.3}  R2={:.3}", m_24_test.rmse, m_24_test.mae, m_24_test.r2);

PERSISTENCE-24h (val):  RMSE=3.328  MAE=2.318  R2=0.887


PERSISTENCE-24h (test): RMSE=4.077  MAE=2.962  R2=0.806


### 10.3 Per-city hourly climatology

For each `(city, hour)` we compute the mean temperature on the train set
and predict that value.

In [17]:
// Table: (city, hour) -> mean(temp_next_24h)
let train_clean_climo = train_clean.clone().lazy()
    .group_by([col("city"), col("hour")])
    .agg([col("temp_next_24h").mean().alias("climo")])
    .collect().unwrap();

let mut climo_table: HashMap<(String, i32), f64> = HashMap::new();
{
    let cities = train_clean_climo.column("city").unwrap().str().unwrap();
    let hours  = train_clean_climo.column("hour").unwrap().i32().unwrap();
    let climo  = train_clean_climo.column("climo").unwrap().f64().unwrap();
    for i in 0..train_clean_climo.height() {
        if let (Some(c), Some(h), Some(v)) = (cities.get(i), hours.get(i), climo.get(i)) {
            climo_table.insert((c.to_string(), h), v);
        }
    }
}

fn predict_climo(df: &DataFrame, table: &HashMap<(String, i32), f64>, fallback: f64) -> Vec<f64> {
    let n = df.height();
    let cities = df.column("city").unwrap().str().unwrap();
    let hours  = df.column("hour").unwrap().i32().unwrap();
    let mut out = Vec::with_capacity(n);
    for i in 0..n {
        let key = (cities.get(i).unwrap_or("").to_string(), hours.get(i).unwrap_or(0));
        out.push(*table.get(&key).unwrap_or(&fallback));
    }
    out
}

let global_mean = y_train_temp24.mean().unwrap_or(0.0);
let p_climo_val  = predict_climo(&val_clean,  &climo_table, global_mean);
let p_climo_test = predict_climo(&test_clean, &climo_table, global_mean);

let m_climo_val  = reg_metrics(y_val_temp24.as_slice().unwrap(),  &p_climo_val);
let m_climo_test = reg_metrics(y_test_temp24.as_slice().unwrap(), &p_climo_test);
println!("CLIMATOLOGY (val):  RMSE={:.3}  MAE={:.3}  R2={:.3}", m_climo_val.rmse, m_climo_val.mae, m_climo_val.r2);
println!("CLIMATOLOGY (test): RMSE={:.3}  MAE={:.3}  R2={:.3}", m_climo_test.rmse, m_climo_test.mae, m_climo_test.r2);

CLIMATOLOGY (val):  RMSE=8.130  MAE=6.086  R2=0.323


CLIMATOLOGY (test): RMSE=7.061  MAE=5.479  R2=0.418


### 10.4 Trivial-majority (classification)

In [18]:
let positives: usize = y_train_rain.iter().filter(|&&v| v == 1).count();
let majority: u32 = if 2*positives > y_train_rain.len() { 1 } else { 0 };
println!("Train-set majority class: {} ({:.1}%)", majority,
    100.0 * positives as f64 / y_train_rain.len() as f64);

let trivial_val:  Vec<u32> = vec![majority; y_val_rain.len()];
let trivial_test: Vec<u32> = vec![majority; y_test_rain.len()];

let m_triv_val  = cls_metrics(&y_val_rain,  &trivial_val);
let m_triv_test = cls_metrics(&y_test_rain, &trivial_test);
println!("TRIVIAL (val):  acc={:.3} balacc={:.3} F1={:.3} MCC={:.3}",
    m_triv_val.accuracy, m_triv_val.bal_acc, m_triv_val.f1, m_triv_val.mcc);
println!("TRIVIAL (test): acc={:.3} balacc={:.3} F1={:.3} MCC={:.3}",
    m_triv_test.accuracy, m_triv_test.bal_acc, m_triv_test.f1, m_triv_test.mcc);

Train-set majority class: 1 (73.7%)


TRIVIAL (val):  acc=0.716 balacc=0.500 F1=0.835 MCC=0.000


TRIVIAL (test): acc=0.710 balacc=0.500 F1=0.830 MCC=0.000


---
## 11. Regression models (target: `temp_next_24h`)

We train each model with reasonable hyperparameters (Notebook 04 will
tune). Every model produces a prediction vector and its metrics are
computed and stored in `Vec<(name, RegMetrics)>` for the final table.

In [19]:
let mut reg_results: Vec<(String, RegMetrics, RegMetrics)> = Vec::new();

// helper to record a model
fn record_reg(results: &mut Vec<(String, RegMetrics, RegMetrics)>,
              name: &str,
              y_val_pred: &[f64], y_val_true: &[f64],
              y_test_pred: &[f64], y_test_true: &[f64]) {
    let mv = reg_metrics(y_val_true, y_val_pred);
    let mt = reg_metrics(y_test_true, y_test_pred);
    println!("{:<28} val RMSE={:.3}  MAE={:.3}  R2={:.3}  | test RMSE={:.3}  MAE={:.3}  R2={:.3}",
             name, mv.rmse, mv.mae, mv.r2, mt.rmse, mt.mae, mt.r2);
    results.push((name.to_string(), mv, mt));
}

// === Baselines (already computed) ===
record_reg(&mut reg_results, "Persistence-now",
    p_now_val.as_slice().unwrap(), y_val_temp24.as_slice().unwrap(),
    p_now_test.as_slice().unwrap(), y_test_temp24.as_slice().unwrap());
record_reg(&mut reg_results, "Persistence-24h",
    p_24_val.as_slice().unwrap(), y_val_temp24.as_slice().unwrap(),
    p_24_test.as_slice().unwrap(), y_test_temp24.as_slice().unwrap());
record_reg(&mut reg_results, "Climatology(city,hour)",
    &p_climo_val, y_val_temp24.as_slice().unwrap(),
    &p_climo_test, y_test_temp24.as_slice().unwrap());

Persistence-now              val RMSE=2.415  MAE=1.746  R2=0.940  | test RMSE=3.760  MAE=2.516  R2=0.835


Persistence-24h              val RMSE=3.328  MAE=2.318  R2=0.887  | test RMSE=4.077  MAE=2.962  R2=0.806


Climatology(city,hour)       val RMSE=8.130  MAE=6.086  R2=0.323  | test RMSE=7.061  MAE=5.479  R2=0.418


### 11.1 Linear models (on standardized features)

In [20]:
let y_train_temp_v: Vec<f64> = y_train_temp24.to_vec();

// 1) OLS
let lr = LinearRegression::fit(&X_train_dm_z, &y_train_temp_v, LinearRegressionParameters::default()).unwrap();
let pred_lr_val:  Vec<f64> = lr.predict(&X_val_dm_z).unwrap();
let pred_lr_test: Vec<f64> = lr.predict(&X_test_dm_z).unwrap();
record_reg(&mut reg_results, "OLS LinearRegression",
    &pred_lr_val, y_val_temp24.as_slice().unwrap(),
    &pred_lr_test, y_test_temp24.as_slice().unwrap());

// 2) Ridge alpha=1.0
let rr = RidgeRegression::fit(&X_train_dm_z, &y_train_temp_v,
    RidgeRegressionParameters::default().with_alpha(1.0)).unwrap();
let pred_rr_val:  Vec<f64> = rr.predict(&X_val_dm_z).unwrap();
let pred_rr_test: Vec<f64> = rr.predict(&X_test_dm_z).unwrap();
record_reg(&mut reg_results, "Ridge (alpha=1.0)",
    &pred_rr_val, y_val_temp24.as_slice().unwrap(),
    &pred_rr_test, y_test_temp24.as_slice().unwrap());

// 3) Lasso alpha=0.1
let lasso = Lasso::fit(&X_train_dm_z, &y_train_temp_v,
    LassoParameters::default().with_alpha(0.1)).unwrap();
let pred_lasso_val:  Vec<f64> = lasso.predict(&X_val_dm_z).unwrap();
let pred_lasso_test: Vec<f64> = lasso.predict(&X_test_dm_z).unwrap();
record_reg(&mut reg_results, "Lasso (alpha=0.1)",
    &pred_lasso_val, y_val_temp24.as_slice().unwrap(),
    &pred_lasso_test, y_test_temp24.as_slice().unwrap());

OLS LinearRegression         val RMSE=2.513  MAE=1.875  R2=0.935  | test RMSE=3.533  MAE=2.433  R2=0.854


Ridge (alpha=1.0)            val RMSE=2.511  MAE=1.874  R2=0.935  | test RMSE=3.529  MAE=2.432  R2=0.855


Lasso (alpha=0.1)            val RMSE=2.383  MAE=1.740  R2=0.942  | test RMSE=3.598  MAE=2.426  R2=0.849


### 11.2 KNN (on standardized features)

In [21]:
// KNN k=15 (conservative; tuning in Nb04)
let knn = KNNRegressor::fit(&X_train_dm_z, &y_train_temp_v,
    KNNRegressorParameters::default().with_k(15)).unwrap();
let pred_knn_val:  Vec<f64> = knn.predict(&X_val_dm_z).unwrap();
let pred_knn_test: Vec<f64> = knn.predict(&X_test_dm_z).unwrap();
record_reg(&mut reg_results, "KNN (k=15)",
    &pred_knn_val, y_val_temp24.as_slice().unwrap(),
    &pred_knn_test, y_test_temp24.as_slice().unwrap());

KNN (k=15)                   val RMSE=3.178  MAE=2.345  R2=0.897  | test RMSE=3.984  MAE=2.798  R2=0.815


### 11.3 Decision Tree (raw features -- scale is irrelevant)

In [22]:
let dt = DecisionTreeRegressor::fit(&X_train_dm, &y_train_temp_v,
    DecisionTreeRegressorParameters::default().with_max_depth(12)).unwrap();
let pred_dt_val:  Vec<f64> = dt.predict(&X_val_dm).unwrap();
let pred_dt_test: Vec<f64> = dt.predict(&X_test_dm).unwrap();
record_reg(&mut reg_results, "DecisionTree (depth=12)",
    &pred_dt_val, y_val_temp24.as_slice().unwrap(),
    &pred_dt_test, y_test_temp24.as_slice().unwrap());

DecisionTree (depth=12)      val RMSE=3.020  MAE=2.157  R2=0.907  | test RMSE=4.192  MAE=2.875  R2=0.795


### 11.4 Random Forest

In [23]:
let rf = RandomForestRegressor::fit(&X_train_dm, &y_train_temp_v,
    RandomForestRegressorParameters::default()
        .with_n_trees(50)
        .with_max_depth(15)
).unwrap();
let pred_rf_val:  Vec<f64> = rf.predict(&X_val_dm).unwrap();
let pred_rf_test: Vec<f64> = rf.predict(&X_test_dm).unwrap();
record_reg(&mut reg_results, "RandomForest (n=50,d=15)",
    &pred_rf_val, y_val_temp24.as_slice().unwrap(),
    &pred_rf_test, y_test_temp24.as_slice().unwrap());

RandomForest (n=50,d=15)     val RMSE=2.468  MAE=1.805  R2=0.938  | test RMSE=3.652  MAE=2.485  R2=0.844


### 11.5 Gradient Boosting (manual — Friedman 2001)

Classical boosting:
$$
F_0(x) = \bar{y},\quad F_k(x) = F_{k-1}(x) + \eta \cdot h_k(x)
$$
where $h_k$ is a shallow tree fit to the residuals $r_k = y - F_{k-1}(x)$.
We use $\eta = 0.1$, $K = 80$ trees, depth $5$ — a stable default for
small/medium tabular datasets.

In [24]:
let n_train = X_train_f.nrows();
let n_val   = X_val_f.nrows();
let n_test  = X_test_f.nrows();
let init_pred = y_train_temp_v.iter().sum::<f64>() / n_train as f64;
let n_trees = 80usize;
let lr_gb = 0.1_f64;
let max_d_gb: u16 = 5;

let mut gb_train_pred: Vec<f64> = vec![init_pred; n_train];
let mut gb_val_pred:   Vec<f64> = vec![init_pred; n_val];
let mut gb_test_pred:  Vec<f64> = vec![init_pred; n_test];

println!("Training Gradient Boosting (K={} trees)...", n_trees);
for k in 0..n_trees {
    let residuals: Vec<f64> = y_train_temp_v.iter().zip(gb_train_pred.iter())
        .map(|(y, p)| y - p).collect();
    let tree = DecisionTreeRegressor::fit(&X_train_dm, &residuals,
        DecisionTreeRegressorParameters::default().with_max_depth(max_d_gb)).unwrap();
    let upd_train: Vec<f64> = tree.predict(&X_train_dm).unwrap();
    let upd_val:   Vec<f64> = tree.predict(&X_val_dm).unwrap();
    let upd_test:  Vec<f64> = tree.predict(&X_test_dm).unwrap();
    for i in 0..n_train { gb_train_pred[i] += lr_gb * upd_train[i]; }
    for i in 0..n_val   { gb_val_pred[i]   += lr_gb * upd_val[i]; }
    for i in 0..n_test  { gb_test_pred[i]  += lr_gb * upd_test[i]; }
    if (k+1) % 20 == 0 {
        let m = reg_metrics(y_val_temp24.as_slice().unwrap(), &gb_val_pred);
        println!("  iter {:>3}: val RMSE={:.3}", k+1, m.rmse);
    }
}

record_reg(&mut reg_results, "GradientBoosting (K=80,d=5,eta=0.1)",
    &gb_val_pred, y_val_temp24.as_slice().unwrap(),
    &gb_test_pred, y_test_temp24.as_slice().unwrap());

Training Gradient Boosting (K=80 trees)...


  iter  20: val RMSE=2.655


  iter  40: val RMSE=2.401


  iter  60: val RMSE=2.392


  iter  80: val RMSE=2.373


GradientBoosting (K=80,d=5,eta=0.1) val RMSE=2.373  MAE=1.737  R2=0.942  | test RMSE=3.640  MAE=2.457  R2=0.845


---
## 12. Classification models (target: `will_rain_next_24h`)

In [25]:
let mut cls_results: Vec<(String, ClsMetrics, ClsMetrics)> = Vec::new();

fn record_cls(results: &mut Vec<(String, ClsMetrics, ClsMetrics)>,
              name: &str,
              y_val_pred: &[u32], y_val_true: &[u32],
              y_test_pred: &[u32], y_test_true: &[u32]) {
    let mv = cls_metrics(y_val_true, y_val_pred);
    let mt = cls_metrics(y_test_true, y_test_pred);
    println!("{:<32} val acc={:.3} balacc={:.3} F1={:.3} MCC={:.3}  | test acc={:.3} balacc={:.3} F1={:.3} MCC={:.3}",
             name, mv.accuracy, mv.bal_acc, mv.f1, mv.mcc, mt.accuracy, mt.bal_acc, mt.f1, mt.mcc);
    results.push((name.to_string(), mv, mt));
}

record_cls(&mut cls_results, "Trivial-majority", &trivial_val, &y_val_rain, &trivial_test, &y_test_rain);

// 1) Logistic Regression
let log = LogisticRegression::fit(&X_train_dm_z, &y_train_rain, LogisticRegressionParameters::default()).unwrap();
let p_log_val:  Vec<u32> = log.predict(&X_val_dm_z).unwrap();
let p_log_test: Vec<u32> = log.predict(&X_test_dm_z).unwrap();
record_cls(&mut cls_results, "LogisticRegression",
    &p_log_val, &y_val_rain, &p_log_test, &y_test_rain);

// 2) KNN Classifier (k=15)
let knnc = KNNClassifier::fit(&X_train_dm_z, &y_train_rain,
    KNNClassifierParameters::default().with_k(15)).unwrap();
let p_knnc_val:  Vec<u32> = knnc.predict(&X_val_dm_z).unwrap();
let p_knnc_test: Vec<u32> = knnc.predict(&X_test_dm_z).unwrap();
record_cls(&mut cls_results, "KNN-Classifier (k=15)",
    &p_knnc_val, &y_val_rain, &p_knnc_test, &y_test_rain);

// 3) Decision Tree Classifier
let dtc = DecisionTreeClassifier::fit(&X_train_dm, &y_train_rain,
    DecisionTreeClassifierParameters::default().with_max_depth(12)).unwrap();
let p_dtc_val:  Vec<u32> = dtc.predict(&X_val_dm).unwrap();
let p_dtc_test: Vec<u32> = dtc.predict(&X_test_dm).unwrap();
record_cls(&mut cls_results, "DecisionTreeClassifier (d=12)",
    &p_dtc_val, &y_val_rain, &p_dtc_test, &y_test_rain);

// 4) Random Forest Classifier
let rfc = RandomForestClassifier::fit(&X_train_dm, &y_train_rain,
    RandomForestClassifierParameters::default().with_n_trees(50).with_max_depth(15)).unwrap();
let p_rfc_val:  Vec<u32> = rfc.predict(&X_val_dm).unwrap();
let p_rfc_test: Vec<u32> = rfc.predict(&X_test_dm).unwrap();
record_cls(&mut cls_results, "RandomForestClassifier (n=50)",
    &p_rfc_val, &y_val_rain, &p_rfc_test, &y_test_rain);

// 5) Gaussian Naive Bayes
let gnb = GaussianNB::fit(&X_train_dm_z, &y_train_rain, GaussianNBParameters::default()).unwrap();
let p_gnb_val:  Vec<u32> = gnb.predict(&X_val_dm_z).unwrap();
let p_gnb_test: Vec<u32> = gnb.predict(&X_test_dm_z).unwrap();
record_cls(&mut cls_results, "GaussianNB",
    &p_gnb_val, &y_val_rain, &p_gnb_test, &y_test_rain);

Trivial-majority                 val acc=0.716 balacc=0.500 F1=0.835 MCC=0.000  | test acc=0.710 balacc=0.500 F1=0.830 MCC=0.000


LogisticRegression               val acc=0.831 balacc=0.792 F1=0.882 MCC=0.583  | test acc=0.845 balacc=0.784 F1=0.895 MCC=0.608


KNN-Classifier (k=15)            val acc=0.830 balacc=0.768 F1=0.885 MCC=0.565  | test acc=0.805 balacc=0.738 F1=0.868 MCC=0.506


DecisionTreeClassifier (d=12)    val acc=0.881 balacc=0.844 F1=0.918 MCC=0.703  | test acc=0.852 balacc=0.803 F1=0.898 MCC=0.630


RandomForestClassifier (n=50)    val acc=0.879 balacc=0.845 F1=0.916 MCC=0.699  | test acc=0.875 balacc=0.832 F1=0.914 MCC=0.688


GaussianNB                       val acc=0.604 balacc=0.711 F1=0.626 MCC=0.400  | test acc=0.579 balacc=0.693 F1=0.586 MCC=0.378


---
## 13. Skill scores vs Persistence-24h and Trivial

In [26]:
let baseline_rmse_test = m_24_test.rmse;
println!("=== SKILL SCORE (regression, vs Persistence-24h) ===");
println!("{:<32} {:>10}", "model", "SS_test");
for (name, _, mt) in &reg_results {
    let ss = 1.0 - (mt.rmse / baseline_rmse_test).powi(2);
    println!("{:<32} {:>10.4}", name, ss);
}

println!();
let baseline_mcc_test = m_triv_test.mcc;
println!("=== SKILL (classification, delta MCC vs Trivial) ===");
println!("{:<32} {:>10} {:>10}", "model", "MCC_test", "delta");
for (name, _, mt) in &cls_results {
    println!("{:<32} {:>10.4} {:>10.4}", name, mt.mcc, mt.mcc - baseline_mcc_test);
}

=== SKILL SCORE (regression, vs Persistence-24h) ===


model                               SS_test


Persistence-now                      0.1491


Persistence-24h                      0.0000


Climatology(city,hour)              -2.0003


...

KNN-Classifier (k=15)                0.5061     0.5061


DecisionTreeClassifier (d=12)        0.6304     0.6304


RandomForestClassifier (n=50)        0.6877     0.6877


GaussianNB                           0.3782     0.3782


()

---
## 14. Final ranking

In [27]:
// Sort regressors by test RMSE ascending
let mut sorted_reg = reg_results.clone();
sorted_reg.sort_by(|a, b| a.2.rmse.partial_cmp(&b.2.rmse).unwrap());

println!("=== FINAL RANKING - REGRESSION (test RMSE ascending) ===");
println!("{:<32} {:>8} {:>8} {:>8} {:>8} {:>8} {:>8}",
         "model", "RMSE", "MAE", "MedAE", "MAPE%", "MBE", "R2");
println!("{}", "-".repeat(82));
for (name, _, mt) in &sorted_reg {
    println!("{:<32} {:>8.3} {:>8.3} {:>8.3} {:>8.2} {:>+8.3} {:>8.3}",
             name, mt.rmse, mt.mae, mt.medae, mt.mape, mt.mbe, mt.r2);
}

let mut sorted_cls = cls_results.clone();
sorted_cls.sort_by(|a, b| b.2.mcc.partial_cmp(&a.2.mcc).unwrap());

println!();
println!("=== FINAL RANKING - CLASSIFICATION (test MCC descending) ===");
println!("{:<32} {:>8} {:>8} {:>8} {:>8} {:>8}",
         "model", "Acc", "BalAcc", "Prec", "Recall", "F1");
println!("{}", "-".repeat(76));
for (name, _, mt) in &sorted_cls {
    println!("{:<32} {:>8.3} {:>8.3} {:>8.3} {:>8.3} {:>8.3}",
             name, mt.accuracy, mt.bal_acc, mt.precision, mt.recall, mt.f1);
}

=== FINAL RANKING - REGRESSION (test RMSE ascending) ===


model                                RMSE      MAE    MedAE    MAPE%      MBE       R2


----------------------------------------------------------------------------------


Ridge (alpha=1.0)                   3.529    2.432    1.633    24.84   -0.688    0.855


OLS LinearRegression                3.533    2.433    1.631    24.81   -0.688    0.854


...

LogisticRegression                  0.845    0.784    0.862    0.930    0.895


KNN-Classifier (k=15)               0.805    0.738    0.838    0.900    0.868


GaussianNB                          0.579    0.693    0.966    0.421    0.586


Trivial-majority                    0.710    0.500    0.710    1.000    0.830


()

---
## 15. Persist comparison results

In [28]:
use serde_json::json;

let reg_json: Vec<_> = reg_results.iter().map(|(name, mv, mt)| {
    json!({
        "model": name,
        "val":  { "rmse": mv.rmse, "mae": mv.mae, "r2": mv.r2, "mape": mv.mape, "mbe": mv.mbe },
        "test": { "rmse": mt.rmse, "mae": mt.mae, "r2": mt.r2, "mape": mt.mape, "mbe": mt.mbe },
    })
}).collect();

let cls_json: Vec<_> = cls_results.iter().map(|(name, mv, mt)| {
    json!({
        "model": name,
        "val":  { "accuracy": mv.accuracy, "balanced_accuracy": mv.bal_acc, "f1": mv.f1, "mcc": mv.mcc,
                  "precision": mv.precision, "recall": mv.recall },
        "test": { "accuracy": mt.accuracy, "balanced_accuracy": mt.bal_acc, "f1": mt.f1, "mcc": mt.mcc,
                  "precision": mt.precision, "recall": mt.recall,
                  "confusion": { "tn": mt.tn, "fp": mt.fp, "fn": mt.fn_, "tp": mt.tp } },
    })
}).collect();

let bundle = json!({
    "n_features": final_features.len(),
    "feature_names": final_features,
    "regression": reg_json,
    "classification": cls_json,
    "best_regression": sorted_reg.first().map(|(n, _, _)| n.clone()),
    "best_classification": sorted_cls.first().map(|(n, _, _)| n.clone()),
});

std::fs::create_dir_all("../models").unwrap();
std::fs::write("../models/model_comparison.json",
    serde_json::to_string_pretty(&bundle).unwrap()).unwrap();
println!("Saved ../models/model_comparison.json");

Saved ../models/model_comparison.json


---
## 16. Climatology export (production fallback)

If the Open-Meteo API fails at prediction time, the production binary
must still output *something*. We export the per-`(city, hour)` mean
temperature (from the Nb03 climatology baseline) so that `src/bin/` can
fall back to it gracefully.

The JSON has shape `{ "city_name": [h0, h1, ..., h23] }` for easy
consumption.

In [29]:
// Build a clean {city: [h0..h23]} structure with global-mean defaults.
let mut climo_out: std::collections::BTreeMap<String, Vec<f64>> = std::collections::BTreeMap::new();
let global_mean_out = global_mean;

let all_cities: Vec<String> = {
    let s = train_clean.column("city").unwrap().str().unwrap();
    let mut set: std::collections::HashSet<String> = std::collections::HashSet::new();
    for v in s.into_iter() { if let Some(c) = v { set.insert(c.to_string()); } }
    let mut v: Vec<String> = set.into_iter().collect();
    v.sort();
    v
};

for city in &all_cities {
    let mut hours: Vec<f64> = Vec::with_capacity(24);
    for h in 0..24_i32 {
        let key = (city.clone(), h);
        hours.push(*climo_table.get(&key).unwrap_or(&global_mean_out));
    }
    climo_out.insert(city.clone(), hours);
}

let climo_json = json!({
    "version": "1.0.0",
    "generator": "Notebook 03",
    "description": "Per-(city, hour) mean of temp_next_24h from the train split",
    "target":     "temp_next_24h",
    "global_mean": global_mean_out,
    "hours_per_row": 24,
    "climatology": climo_out,
});

std::fs::write("../models/climatology.json",
    serde_json::to_string_pretty(&climo_json).unwrap()).unwrap();
println!("Saved ../models/climatology.json ({} cities, 24 hours each)", all_cities.len());

Saved ../models/climatology.json (14 cities, 24 hours each)


---
## 17. Summary

- **Notebook 04** will take the top 2-3 models from this table and run
  grid/random search + time-series cross-validation.
- **Notebook 05** does the rigorous final evaluation on the held-out test
  set with per-city/season/temperature-bin breakdowns.
- **Notebook 06** monitors drift for the **chosen model**.

New artifact in this revision: `models/climatology.json` (fallback table).

In [30]:
println!("\n{}", "=".repeat(60));
println!("Notebook 03 complete.");
println!("{}", "=".repeat(60));
println!("Next: Notebook 04 - Hyperparameter Tuning (TimeSeriesSplit, grid + random search)");

Notebook 03 complete.


Next: Notebook 04 - Hyperparameter Tuning (TimeSeriesSplit, grid + random search)
